### Links and Resources
| Function | Use |
|---|---|
| [`array`, `array_contains`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.array_contains.html) | Build / probe arrays |
| [`size`, `array_distinct`, `sort_array`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.size.html) | Array shape & cleanup |
| [`explode`, `posexplode`, `explode_outer`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html) | Array → rows |
| [`flatten`, `arrays_zip`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.flatten.html) | Multi-array combinators |
| [`sequence`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.sequence.html) | Generate ranges |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Collection (Array) Functions

Spark arrays are first-class column types. They show up everywhere — JSON ingestion, `split` results, multi-tag fields, time-series buckets. Most of the work is one of three things: **build**, **probe**, or **fan out**.

In [ ]:
df = spark.createDataFrame([
    (1, "Alice",   ["python", "sql", "spark"]),
    (2, "Bob",     ["scala", "spark"]),
    (3, "Charlie", []),
    (4, "Dana",    None),
    (5, "Erin",    ["python", "python", "rust"]),
], ["id", "name", "skills"])
display(df)

## Build — `array`, `lit`, `sequence`

In [ ]:
spark.range(3).select(
    "id",
    F.array(F.lit("a"), F.lit("b"), F.col("id").cast("string")).alias("arr"),
    F.sequence(F.lit(1), F.col("id") + 2).alias("range"),
).show(truncate=False)

## Shape — `size`, `array_distinct`, `sort_array`, `array_max`, `array_min`

`size` returns -1 for NULL arrays and 0 for empty — distinguish them with `isNull()`.

In [ ]:
df.select(
    "name",
    F.size("skills").alias("n"),
    F.array_distinct("skills").alias("unique"),
    F.sort_array("skills").alias("sorted"),
).show(truncate=False)

## Probe — `array_contains`, `array_position`

Both return useful values for filtering.

In [ ]:
df.filter(F.array_contains("skills", "spark")).select("name").show()

df.select(
    "name",
    F.array_position("skills", "python").alias("python_pos"),  # 1-based, 0 if absent
).show()

## Set ops — `array_union`, `array_intersect`, `array_except`

In [ ]:
pairs = df.select("name", "skills").alias("a").crossJoin(
    df.select(F.col("name").alias("other"), F.col("skills").alias("other_skills"))
).filter("name < other").limit(3)

pairs.select(
    "name", "other",
    F.array_intersect("skills", "other_skills").alias("shared"),
    F.array_except("skills", "other_skills").alias("only_in_a"),
).show(truncate=False)

## Fan out — `explode`, `explode_outer`, `posexplode`

`explode` drops rows where the array is NULL or empty. `explode_outer` keeps them with a NULL value. `posexplode` adds a position column.

In [ ]:
print("explode (drops NULL/empty):")
df.select("name", F.explode("skills").alias("skill")).show()

print("explode_outer (keeps them):")
df.select("name", F.explode_outer("skills").alias("skill")).show()

print("posexplode (with index):")
df.select("name", F.posexplode("skills").alias("idx", "skill")).show()

## Combinators — `flatten`, `arrays_zip`

`flatten` collapses one level of nesting. `arrays_zip` merges parallel arrays element-wise into an array of structs.

In [ ]:
nested = spark.createDataFrame([
    (1, [["a", "b"], ["c"]],     ["x", "y", "z"], [10, 20, 30]),
], ["id", "nested", "letters", "numbers"])

nested.select(
    F.flatten("nested").alias("flat"),
    F.arrays_zip("letters", "numbers").alias("zipped"),
).show(truncate=False)

## NULL semantics

- `size(NULL) = -1`, `size([]) = 0`. Use `isNull()` to tell them apart.
- `array_contains(NULL, "x")` returns NULL, not False — wrap with `coalesce(..., F.lit(False))` if you filter on it.

In [ ]:
# Safe filter that treats NULL arrays as "doesn't contain"
df.filter(
    F.coalesce(F.array_contains("skills", "spark"), F.lit(False))
).select("name").show()

## Takeaways

- `array` builds, `array_contains` / `array_position` probe, `explode` fans out.
- `size` distinguishes empty (0) from NULL (-1).
- Use `_outer` variants of `explode` to preserve rows with NULL/empty arrays.
- Set operations (`union`, `intersect`, `except`) work element-wise on arrays.
- For *element-wise transformations* (e.g. uppercase every string in an array) you want **higher-order functions** — see notebook 08.